# Faithful-run monitor

Set **BOOK / PROP** in the next cell, then **Run All**. Re-run any cell to refresh while a `run_faithful.py` job is going. Shows cost per phase, cumulative cost over prove sessions, and the newest trace (thinking / tools / output).

In [24]:
import os, json, glob
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

# ==== set these, then Run All; re-run to refresh ====
BOOK, PROP = 1, 8

ROOT = r"/h/56/taddmao/code/autoform/DNA/LeanEuclidPlus"
PROPDIR = os.path.join(ROOT, "Book%d" % BOOK, "Prop%02d" % PROP)
print(PROPDIR, "\n(exists:", os.path.isdir(PROPDIR), ")")

/h/56/taddmao/code/autoform/DNA/LeanEuclidPlus/Book1/Prop08 
(exists: True )


In [25]:
def _load(p):
    return json.load(open(p)) if os.path.exists(p) else None

cost = os.path.join(PROPDIR, "cost")
for ph in ("split", "translate"):
    r = _load(os.path.join(cost, ph + ".json"))
    if r:
        print("%-9s $%.4f  [%s]" % (ph, r.get("cost_usd") or 0, r.get("model", "?")))

prove = sorted(glob.glob(os.path.join(cost, "prove", "*.json")),
               key=lambda p: int(os.path.basename(p)[:-5]) if os.path.basename(p)[:-5].isdigit() else 0)
recs = [_load(p) for p in prove]
for r in recs:
    nodes = r.get("status", {}).get("nodes", [])
    done = sum(1 for n in nodes if n.get("state") == "done")
    print("prove #%s  $%.4f  cum $%.4f  %d/%d nodes done"
          % (r.get("seq"), r.get("this_session_usd") or 0, r.get("cumulative_usd") or 0, done, len(nodes)))

summ = _load(os.path.join(cost, "summary.json"))
if summ:
    print("TOTAL  $%.4f" % (summ.get("total_usd") or 0))

if recs:
    xs = [r.get("seq") for r in recs]
    ys = [r.get("cumulative_usd") or 0 for r in recs]
    plt.figure(figsize=(6, 3))
    plt.plot(xs, ys, marker="o")
    plt.xlabel("prove session"); plt.ylabel("cumulative $")
    plt.title("Book%d/Prop%02d prove cost over time" % (BOOK, PROP))
    plt.grid(True); plt.show()

In [26]:
def _newest_trace(propdir):
    fs = glob.glob(os.path.join(propdir, "runs", "*.jsonl"))
    return max(fs, key=os.path.getmtime) if fs else None

def render_trace(path):
    md = ["### trace `%s`" % os.path.basename(path)]
    for line in open(path):
        line = line.strip()
        if not line.startswith("{"):
            continue
        try:
            o = json.loads(line)
        except ValueError:
            continue
        t = o.get("type")
        if t == "system":
            md.append("_session %s \u00b7 model %s_" % (str(o.get("session_id"))[:8], o.get("model")))
        elif t == "assistant":
            for b in (o.get("message") or {}).get("content") or []:
                bt = b.get("type")
                if bt == "thinking":
                    md.append("> \U0001f4ad " + " ".join(b.get("thinking", "").split()))
                elif bt == "text" and b.get("text", "").strip():
                    md.append(b.get("text"))
                elif bt == "tool_use":
                    md.append("**\u25b6 %s** `%s`" % (b.get("name"), json.dumps(b.get("input"))[:220]))
        elif t == "result":
            c = o.get("total_cost_usd") or (o.get("usage") or {}).get("total_cost_usd") or 0
            md.append("**done** \u2014 $%.4f \u00b7 %s turns" % (c, o.get("num_turns")))
    display(Markdown(("\n\n").join(md)))

tr = _newest_trace(PROPDIR)
print(tr)
if tr:
    render_trace(tr)

/h/56/taddmao/code/autoform/DNA/LeanEuclidPlus/Book1/Prop08/runs/split-1.jsonl


### trace `split-1.jsonl`

_session 768a53bb · model claude-opus-4-8_

> 💭 

I'll read the Book 1 text file to find Proposition 8.

**▶ Bash** `{"command": "realpath LeanEuclidPlus/Book1/data/texts_proofs/1.txt", "description": "Resolve path to Book 1 text file"}`

**▶ Read** `{"file_path": "/h/56/taddmao/code/autoform/DNA/LeanEuclidPlus/Book1/data/texts_proofs/1.txt"}`

> 💭 

This is Prop 1. Let me find the correct text file structure for Prop 8.

**▶ Bash** `{"command": "ls LeanEuclidPlus/Book1/data/texts_proofs/", "description": "List Book 1 text proof files"}`

**▶ Glob** `{"pattern": "LeanEuclidPlus/Book1/data/texts_proofs/*"}`

**▶ Bash** `{"command": "git ls-files LeanEuclidPlus/Book1/data/texts_proofs/", "description": "List tracked text proof files"}`

**▶ Read** `{"file_path": "/h/56/taddmao/code/autoform/DNA/LeanEuclidPlus/Book1/data/texts_proofs/8.txt"}`